In [1]:

# from pyspark.sql import SparkSession

# spark = SparkSession.builder \
#     .appName("RetailPipeline") \
#     .master("local[*]") \
#     .getOrCreate()


ERROR StatusLogger Reconfiguration failed: No configuration found for '5c29bfd' at 'null' in 'null'
ERROR StatusLogger Reconfiguration failed: No configuration found for 'Default' at 'null' in 'null'
26/04/20 22:32:51 WARN Utils: Your hostname, YTuSurface5 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/20 22:32:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 22:32:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [1]:
# need install delta spark - version compatibal e.g. spark 3.5.x with delta spark v3.X
from delta import *
from pyspark.sql import SparkSession

builder = SparkSession.builder \
    .appName("RetailPipeline") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

ERROR StatusLogger Reconfiguration failed: No configuration found for '5c29bfd' at 'null' in 'null'
ERROR StatusLogger Reconfiguration failed: No configuration found for 'Default' at 'null' in 'null'
26/04/20 23:00:05 WARN Utils: Your hostname, YTuSurface5 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/20 23:00:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/yiqingtu/spark/spark-3.5.5-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/yiqingtu/.ivy2/cache
The jars for the packages stored in: /home/yiqingtu/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fcbcc8ea-5a18-40ce-8a7d-4e0e1031a449;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (525ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (121ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (126ms)
:: resolution report :: resolve 3042m

In [8]:
from pyspark.sql.functions import col

file_name = "../data/raw/orders/orders_2014-01-04.csv"

df = spark.read.csv(file_name, header=True, inferSchema=True)

df.limit(1000).toPandas()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,subcategory,product_name,sales,quantity,discount,profit
0,740,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
1,741,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
2,742,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870


In [21]:
# df.write.format("delta") \
#   .mode("append") \
#   .save("./data/delta/bronze/orders")

# use path vs use register table - record in catalog

df.write.format("delta").saveAsTable("bronze_orders")


In [23]:
# test -> read delta table back 

# check count
# print schema
# check physical location use ls 

check_df = spark.read.table("bronze_orders")
check_df.limit(1000).toPandas()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,subcategory,product_name,sales,quantity,discount,profit
0,740,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
1,741,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
2,742,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870


In [13]:
check_df.count()

6

In [19]:
ls ./data/delta/bronze/orders


_delta_log/
part-00000-65158661-6bd1-45b2-9dc5-e15002bcd883-c000.snappy.parquet
part-00000-f179820a-21ff-4278-b0f7-c515ec81ed89-c000.snappy.parquet


In [26]:
check_df = spark.read.table("bronze_orders")

# need panda library
check_df.limit(1000).toPandas()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,subcategory,product_name,sales,quantity,discount,profit
0,740,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
1,741,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
2,742,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870


In [27]:
spark.table("bronze_orders").printSchema()

root
 |-- row_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)



In [28]:
spark.table("bronze_orders").show(5)


+------+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+----------+--------+-----------+-------+---------------+---------------+-----------+--------------------+-------+--------+--------+--------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|customer_name|    segment|      country|      city|   state|postal_code| region|     product_id|       category|subcategory|        product_name|  sales|quantity|discount|  profit|
+------+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+----------+--------+-----------+-------+---------------+---------------+-----------+--------------------+-------+--------+--------+--------+
|   740|CA-2014-112326|2014-01-04|2014-01-08|Standard Class|   PO-19195|Phillina Ober|Home Office|United States|Naperville|Illinois|      60540|Central|OFF-LA-10003223|Office Supplies|     Labels|           Avery 508| 11.784|       3|     0.